In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeRegressor



In [6]:
df = pd.read_csv('processed_data.csv')

In [7]:
# Hedef değişkeni (örneğin, 'charges') ve özellikleri ayırma
X = df.drop('charges', axis=1)  # Bağımsız değişkenler
y = df['charges']              # Bağımlı değişken

# Eğitim ve test setlerine ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#linear regression

model = LinearRegression()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Model Performansı
print(f"Ortalama Mutlak Hata (MAE): {mae}")
print(f"Ortalama Kare Hata (MSE): {mse}")
print(f"Kök Ortalama Kare Hata (RMSE): {rmse}")
print(f"R^2 Skoru: {r2}")



Ortalama Mutlak Hata (MAE): 4186.508898366438
Ortalama Kare Hata (MSE): 33635210.43117845
Kök Ortalama Kare Hata (RMSE): 5799.5870914383595
R^2 Skoru: 0.7833463107364536


In [38]:
#grid search cv ile random forest
from sklearn.model_selection import GridSearchCV

# Parametreler için bir grid tanımlayın
param_grid = {
    'n_estimators': [100, 200, 300],  # Ağaç sayısı
    'max_depth': [None, 10, 20, 30],  # Ağaç derinliği
    'min_samples_split': [2, 5, 10],  # Bir düğümü ikiye ayırmak için gereken örnek sayısı
    'min_samples_leaf': [1, 2, 4],    # Yaprak düğümdeki minimum örnek sayısı
    'max_features': [None, 'sqrt', 'log2'],  # En iyi bölme için kullanılacak özellik sayısı
}

# GridSearchCV ile modelin en iyi hiperparametrelerini bulma
grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42), param_grid=param_grid, 
                           cv=5, n_jobs=-1, verbose=2)

grid_search.fit(X_train, y_train)

# En iyi parametreler
print("En İyi Parametreler:", grid_search.best_params_)

# En iyi model ile tahmin yapma
y_pred_rf_grid = grid_search.best_estimator_.predict(X_test)

# Performans ölçümleri
mse_grid = mean_squared_error(y_test, y_pred_rf_grid)
mae_grid = mean_absolute_error(y_test, y_pred_rf_grid)
rmse_grid = np.sqrt(mse_grid)
r2_grid = r2_score(y_test, y_pred_rf_grid)

# Model Performansı
print(f"Ortalama Mutlak Hata (MAE): {mae_grid}")
print(f"Ortalama Kare Hata (MSE): {mse_grid}")
print(f"Kök Ortalama Kare Hata (RMSE): {rmse_grid}")
print(f"R^2 Skoru: {r2_grid}")


Fitting 5 folds for each of 324 candidates, totalling 1620 fits


c:\Users\tirek\anaconda3\envs\ML\Lib\site-packages\numpy\ma\core.py:2846: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


En İyi Parametreler: {'max_depth': 10, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
Ortalama Mutlak Hata (MAE): 2437.0934842617958
Ortalama Kare Hata (MSE): 18949384.43833512
Kök Ortalama Kare Hata (RMSE): 4353.089068504701
R^2 Skoru: 0.8779417760373828


In [37]:
#randomized searchcv ile random forest
from sklearn.model_selection import RandomizedSearchCV

# Parametreler
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ['sqrt', 'log2'],  # 'auto' yerine 'sqrt' veya 'log2'
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Random Forest model
rf_model = RandomForestRegressor(random_state=42)

# RandomizedSearchCV
random_search = RandomizedSearchCV(rf_model, param_distributions=param_dist, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

# En iyi parametreler
print(f"En İyi Parametreler: {random_search.best_params_}")

# Model Performansı
y_pred_rf = random_search.predict(X_test)
mse = mean_squared_error(y_test, y_pred_rf)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_rf)

# Sonuçlar
print(f"Ortalama Mutlak Hata (MAE): {mae}")
print(f"Ortalama Kare Hata (MSE): {mse}")
print(f"Kök Ortalama Kare Hata (RMSE): {rmse}")
print(f"R^2 Skoru: {r2}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits
En İyi Parametreler: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30}
Ortalama Mutlak Hata (MAE): 2654.0681536728207
Ortalama Kare Hata (MSE): 20321829.509092722
Kök Ortalama Kare Hata (RMSE): 4507.973991616713
R^2 Skoru: 0.8691014779069576


In [36]:

# Decision Tree Modeli
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)  # Eğitim
y_pred_dt = dt_model.predict(X_test)  # Tahmin

# Model Değerlendirmesi
print("Decision Tree Modeli:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_dt)}")
print(f"MSE: {mean_squared_error(y_test, y_pred_dt)}")
print(f"R^2: {r2_score(y_test, y_pred_dt)}")


Decision Tree Modeli:
MAE: 3154.4987263768658
MSE: 49002979.21228644
R^2: 0.6843582634046639


In [42]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Model oluştur
dt_model = DecisionTreeRegressor(random_state=42)

# Parametre ızgarasını tanımla
param_grid = {
    'max_depth': [3, 5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson']
}

# GridSearchCV ile model eğit
grid_search = GridSearchCV(estimator=dt_model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

# En iyi parametreler
print("En iyi parametreler:", grid_search.best_params_)

# En iyi modelin eğitildiği skoru al
best_model = grid_search.best_estimator_
print("En iyi modelin R^2 skoru:", best_model.score(X_test_scaled, y_test))


Fitting 5 folds for each of 540 candidates, totalling 2700 fits
En iyi parametreler: {'criterion': 'absolute_error', 'max_depth': 5, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 2}
En iyi modelin R^2 skoru: 0.8677101158238485


c:\Users\tirek\anaconda3\envs\ML\Lib\site-packages\numpy\ma\core.py:2846: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [43]:
from sklearn.model_selection import RandomizedSearchCV

# Parametre ızgarası
param_dist = {
    'max_depth': [3, 5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [ 'sqrt', 'log2', None],
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson']

}

# RandomizedSearchCV ile model eğit
random_search = RandomizedSearchCV(estimator=dt_model, param_distributions=param_dist, n_iter=100, cv=5, n_jobs=-1, verbose=1, random_state=42)
random_search.fit(X_train_scaled, y_train)

# En iyi parametreler
print("En iyi parametreler:", random_search.best_params_)

# En iyi modelin eğitildiği skoru al
best_model_random = random_search.best_estimator_
print("En iyi modelin R^2 skoru:", best_model_random.score(X_test_scaled, y_test))


Fitting 5 folds for each of 100 candidates, totalling 500 fits
En iyi parametreler: {'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 5, 'criterion': 'absolute_error'}
En iyi modelin R^2 skoru: 0.8677101158238485
